# PLL lock time

Results of `pll_lock_time.lock_time_sweep` (`sw/tests/pll_lock_time.py`): the reference hops by `step_rel` with FSK, the RTB2004 captures the divided VCO on the pad triggered on the FG Sync output, and the VCO frequency against time is averaged over the captures. Per point and step direction (up / down): the settled frequencies, the measured step against `FB_DIV` x the reference step, and the time after the step from which the frequency stays inside each band (10 % / 1 % of the step, 1000 / 100 ppm).

In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
import plotly.io as pio
from plotly.subplots import make_subplots

# VS Code: the default renderer needs the jupyter-renderers extension ("Error loading renderer" when it fails
# to load); plain HTML with plotly.js from the CDN shows without it. HTML and PNG are saved next to the CSV anyway
pio.renderers.default = 'notebook_connected'

PLL_DIR = Path.cwd().resolve().parents[2] / 'results' / 'pll'  # notebook lives in sw/tools/notebooks/
# The run to plot; None = the newest pll_lock_time_*.csv
RUN_CSV = None
# Bands of SETTLE_BANDS (pll_lock_time.py): column suffix -> (kind, value), as in the script
BANDS = {'10pct': ('step', 0.1), '1pct': ('step', 0.01), '1000ppm': ('rel', 1e-3), '100ppm': ('rel', 1e-4)}
ZOOM_US = 3  # [us] after the step shown in the zoomed plot (the PLL settles in ~0.2-0.5 us)


def newest_run():
    return sorted((p for p in PLL_DIR.glob('pll_lock_time_*.csv') if not p.stem.endswith('_traces')),
                  key=lambda p: p.stat().st_mtime)[-1]


RUN_CSV = Path(RUN_CSV) if RUN_CSV is not None else newest_run()
rows = pd.read_csv(RUN_CSV)
traces_path = RUN_CSV.with_name(RUN_CSV.stem + '_traces.csv')
traces = pd.read_csv(traces_path) if traces_path.exists() and traces_path.stat().st_size else pd.DataFrame()
RUN_TXT = RUN_CSV.stem
measured = rows[rows['slope'].notna()].reset_index(drop=True)  # not_locked / error points have no slope

table = pd.DataFrame({
    'point': rows['label'],
    'step': rows['direction'],
    'status': rows['status'],
    'VCO [GHz]': (rows['f_vco'] / 1e9).round(4),
    'codes c/min/max': rows[['vco_tune_coarse', 'vco_current_min', 'vco_current_max']].astype('Int64').astype(str)
                       .agg('/'.join, axis=1),
    'Icp/C1/C2/R1': rows[['set_current', 'set_c1', 'set_c2', 'set_r1']].astype('Int64').astype(str).agg('/'.join, axis=1),
    'locked / hopping / end': rows[['locked', 'locked_fsk', 'locked_end']].astype('Int64').astype(str)
                              .agg('/'.join, axis=1),
    'captures': rows['n_valid'].astype('Int64').astype(str) + '/' + rows['n_captures'].astype('Int64').astype(str),
    'step [MHz]': (rows['step_meas'] / 1e6).round(3),
    'expected [MHz]': (rows['step_expected'] / 1e6).round(3),
    'noise [ppm]': rows['noise_ppm'].round(1),
    **{f't {b} [us]': (rows[f't_settle_{b}'] * 1e6).round(2) for b in BANDS},
    't 10% per capture [us]': ((rows['t_settle_10pct_capt_median'] * 1e6).round(2).astype(str) + ' ± '
                               + (rows['t_settle_10pct_capt_std'] * 1e6).round(2).astype(str)),
    'P [mW]': (rows['pll_p'] * 1e3).round(3),
})
print(f'{RUN_TXT}: {len(rows)} rows, {len(measured)} measured slopes; sample rate '
      f"{rows['sample_rate'].dropna().unique() / 1e6} MSa/s, pad {rows['f_pad'].dropna().unique() / 1e6} MHz")
print('t <band> empty: not settled inside the window (or not measured); a band below ~3x the noise only measures '
      'the noise')
display(table)


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to RUN_CSV (<stem>_<name>), then show the figure."""
    base = RUN_CSV.with_name(f'{RUN_CSV.stem}_{name}')
    fig.write_html(base.with_suffix('.html'))
    try:
        fig.write_image(base.with_suffix('.png'), width=1100, height=fig.layout.height or 600, scale=2)
        print(f'saved {base}.html and .png')
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f'saved {base}.html; PNG export failed: {e}')
    fig.show()


def curve_of(r):
    """The averaged curve of a row (point label and slope), sorted in time."""
    if traces.empty:
        return traces
    return traces[(traces['label'] == r['label']) & (traces['slope'] == r['slope'])].sort_values('t')

pll_lock_time_S5_20261109_143703: 2 rows, 2 measured slopes; sample rate [833.] MSa/s, pad [30.] MHz
t <band> empty: not settled inside the window (or not measured); a band below ~3x the noise only measures the noise


,point,step,status,VCO [GHz],codes c/min/max,Icp/C1/C2/R1,locked / hopping / end,captures,step [MHz],expected [MHz],noise [ppm],t 10pct [us],t 1pct [us],t 1000ppm [us],t 100ppm [us],t 10% per capture [us],P [mW]
0,1/1 4.800 GHz nominal,up,too_noisy,4.8,0/8/8,7/6/0/7,1/1/1,8/8,47.949,48.0,3948.1,NaN,NaN,NaN,NaN,nan ± nan,1.063
1,1/1 4.800 GHz nominal,down,ok,4.8,0/8/8,7/6/0/7,1/1/1,8/8,-46.957,-48.0,191.9,0.2,NaN,0.2,NaN,nan ± nan,1.063


## Settling after the step

VCO frequency minus the settled value (`f_after`) against the time after the Sync edge, averaged over the captures (shaded: ± the spread of the single captures). Left: the whole window; right: the first `ZOOM_US` µs. Dotted: ±10 % and ±1 % of the step; vertical lines: the settling times to those bands.

In [18]:
COLORS = {'up': '#d1495b', 'down': '#2a6fdb'}

for label, point in measured.groupby('label', sort=False):
    fig = make_subplots(rows=1, cols=2, subplot_titles=('whole window', f'first {ZOOM_US} µs'))
    for _, r in point.iterrows():
        c = curve_of(r)
        if c.empty:
            continue
        color, t_us = COLORS.get(r['direction'], 'gray'), c['t'] * 1e6
        err = (c['f_vco_mean'] - r['f_after']) / 1e6
        for col in (1, 2):
            fig.add_trace(go.Scatter(x=pd.concat([t_us, t_us[::-1]]),
                                     y=pd.concat([err + c['f_vco_std'] / 1e6, (err - c['f_vco_std'] / 1e6)[::-1]]),
                                     fill='toself', fillcolor=color, opacity=0.15, line=dict(width=0),
                                     hoverinfo='skip', showlegend=False), row=1, col=col)
            fig.add_trace(go.Scatter(x=t_us, y=err, mode='lines', line=dict(color=color, width=1.5),
                                     name=f"{r['direction']} ({r['step_meas'] / 1e6:+.1f} MHz)", showlegend=col == 1,
                                     hovertemplate='%{x:.2f} us<br>%{y:.3f} MHz<extra></extra>'), row=1, col=col)
            for b, dash in (('10pct', 'dot'), ('1pct', 'dash')):
                t = r[f't_settle_{b}'] * 1e6
                if not np.isnan(t) and (col == 1 or t <= ZOOM_US):
                    fig.add_vline(x=t, line=dict(color=color, dash=dash, width=1), row=1, col=col)
    step = point['step_meas'].abs().max() / 1e6
    for frac in (0.1, 0.01):
        for sign in (1, -1):
            fig.add_hline(y=sign * frac * step, line=dict(color='gray', dash='dot', width=1))
    fig.update_xaxes(title='time after the step [µs]')
    fig.update_xaxes(range=[-0.1 * ZOOM_US, ZOOM_US], row=1, col=2)
    fig.update_yaxes(title='f_vco − settled [MHz]', row=1, col=1)
    fig.update_layout(title=f'Settling, {label} ({RUN_TXT})', height=500, template='plotly_white')
    save(fig, 'settling_' + label.split()[0].replace('/', 'of'))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_143703_settling_1of1.html and .png


## Frequency error on a log scale

|f_vco − settled| / settled [ppm] after the step: where the curve meets the 1000 / 100 ppm lines is the settling time to those bands; the floor it ends on is the measurement noise (`noise_ppm`, dash-dot).

In [19]:
for label, point in measured.groupby('label', sort=False):
    fig = go.Figure()
    for _, r in point.iterrows():
        c = curve_of(r)
        if c.empty:
            continue
        c = c[c['t'] >= 0]
        color = COLORS.get(r['direction'], 'gray')
        fig.add_trace(go.Scatter(x=c['t'] * 1e6, y=(c['f_vco_mean'] - r['f_after']).abs() / r['f_after'] * 1e6,
                                 mode='lines', line=dict(color=color, width=1.5), name=r['direction'],
                                 hovertemplate='%{x:.2f} us<br>%{y:.1f} ppm<extra></extra>'))
        fig.add_hline(y=r['noise_ppm'], line=dict(color=color, dash='dashdot', width=1))
    for b, (kind, value) in BANDS.items():
        if kind == 'rel':
            fig.add_hline(y=value * 1e6, line=dict(color='gray', dash='dot', width=1), annotation_text=b,
                          annotation_position='top right')
    fig.update_xaxes(title='time after the step [µs]')
    fig.update_yaxes(title='|frequency error| [ppm]', type='log', range=[0, 5])  # 1 ppm - 10 %, fixed: hline annotations broke the autorange
    fig.update_layout(title=f'Frequency error, {label} ({RUN_TXT})', height=450, template='plotly_white')
    save(fig, 'error_' + label.split()[0].replace('/', 'of'))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_143703_error_1of1.html and .png


## Lock time per capture

Settling time to 10 % of the step of every single capture (`t_settle_10pct_capt`). A tight cluster: a repeatable linear settling; outliers far above it: cycle slips or a capture where the loop started from an unsettled state. Missing captures did not settle inside the window.

In [20]:
fig = go.Figure()
for _, r in measured.iterrows():
    vals = pd.to_numeric(pd.Series(str(r['t_settle_10pct_capt']).split()), errors='coerce') * 1e6
    fig.add_trace(go.Box(y=vals, name=f"{r['label']} {r['direction']}", boxpoints='all', jitter=0.4, pointpos=0,
                         marker=dict(color=COLORS.get(r['direction'], 'gray'), size=6),
                         line=dict(color=COLORS.get(r['direction'], 'gray'))))
    n_nan = int(vals.isna().sum())
    if n_nan:
        print(f"{r['label']} {r['direction']}: {n_nan} of {len(vals)} captures not settled inside the window")
fig.update_yaxes(title='lock time to 10 % of the step [µs]', rangemode='tozero')
fig.update_layout(title=f'Lock time per capture ({RUN_TXT})', height=450, template='plotly_white', showlegend=False)
save(fig, 'captures')

1/1 4.800 GHz nominal up: 8 of 8 captures not settled inside the window
1/1 4.800 GHz nominal down: 8 of 8 captures not settled inside the window
saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_143703_captures.html and .png
